# Supply Chain Resilience KG — Agentic AI Pipeline

**PDF → Docling Markdown → Semantic Chunks → Ontology-Guided Claude Extraction → Neo4j KG**

This notebook implements an end-to-end pipeline that extracts supply chain resilience entities from corporate documents (annual reports, sustainability reports) and loads them into a Neo4j knowledge graph aligned to a custom OWL/RDF ontology. It also includes an evaluation section with a gold set P/R/F1 evaluation and an ablation study comparing chunking strategies.

**Documents processed:** Hon Hai 2024, Apple Supply Chain 2024 & 2026, Samsung Sustainability 2025, Dixon Technologies 2024-25, Pegatron 2024

**Evaluation:** SHACL validation (84.8%), P/R/F1 on annotated gold set (F1=0.65), 20 Competency Questions via Cypher (10/20 answerable)

---

## ⚠️ How to run this notebook

**Requirements:**
- Python 3.10+
- [Ollama](https://ollama.com) installed and running (for embeddings only)
- Anthropic API key (get one at console.anthropic.com)
- Neo4j AuraDB instance (free tier at neo4j.com/cloud/aura)

1. Clone the repo and move this notebook **one level above** the repo folder. The notebook lives inside `notebooks/full_pipeline/` by default — move it out so your folder structure looks like this:

```
your_workspace/
├── supply_chain_resilience_pipeline.ipynb  ← notebook goes here
├── supply_chain_docs/                       ← put your PDFs here
└── forked_kg_construction/                  ← the cloned repo
    ├── skgb/
    ├── ontology/
    └── requirements.txt
```

2. Create and activate a virtual environment in your terminal:
   - Mac/Linux: `python3 -m venv venv` followed by `source venv/bin/activate`
   - Windows: `python -m venv venv` followed by `venv\Scripts\activate`
3. Run **1.a Setup everything:** — it will clone the repo and install all dependencies automatically.
4. Make sure Ollama is running and `nomic-embed-text` is pulled.
5. Put your PDFs in the `supply_chain_docs/` folder (created automatically if missing).
6. Run all cells top to bottom.

**Every time after that:**
- Activate the venv: `source venv/bin/activate`
- Skip **1.a Setup everything:**
- Run from **1.b Verify Imports** onwards.

---

## 🗺️ Pipeline Flow

**Main Processing Pipeline:**
1. **1.a. Setup everything:** — install dependencies (first time only)
2. **1.b. Verify Imports** — numpy check, API key prompt
3. **2. Start Ollama (embeddings only)** — checks if running, pulls nomic-embed-text if needed
4. **3.a. Logging** — configure logging
5. **3.b. Upload PDF** — scans supply_chain_docs/ for PDFs
6. **4. Configure Pipeline** — set up SKGBConfig
7. **4.b Runtime Estimation (Optional)** — optional chunk count estimate before running
8. **5. Run the Pipeline** — Docling PDF → Markdown → chunks → all_chunks.json
9. **6. Chunks Preview** — sanity check on generated chunks
10. **7. Ontology-Guided Extraction** — Pydantic schema, Claude init, full extraction loop, deduplication
11. **8. Neo4j Loading** — load nodes and relationships into Neo4j AuraDB
12. **9. SHACL Validation** — validate graph against ontology shapes
13. **10. CQ Answerability** — run 20 competency questions via Cypher
14. **11. Explore Results** — summary tables and graph statistics
15. **12. Download Results** — zip and save outputs
16. **13. Stop Ollama** — clean up background processes

**Evaluation Section (optional — for thesis evaluation only):**
- **14–18. Gold Set** — chunk selection, manual annotation comparison, Claude extraction, P/R/F1 scoring
- **19–21. Ablation** — token-aware chunking generation, matched chunk extraction, comparison table

---

**Notes:**
- The `forked_kg_construction/` repo will be cloned automatically if not present.
- The `supply_chain_docs/` folder will be created automatically if not present.
- Neo4j and Anthropic credentials are prompted securely — never hardcoded.
- Extraction costs approximately $0.007 per chunk via Claude Sonnet (~$18-20 for 2500 chunks).
- Gold set evaluation and ablation study cost approximately $0.05 total.

## 1.a. Setup everything:

In [ ]:
import subprocess, sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_PATH = NOTEBOOK_DIR / "forked_kg_construction"

print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Repo path: {REPO_PATH}")

if not REPO_PATH.exists():
    print("📥 Cloning repo...")
    subprocess.run(["git", "clone", "https://github.com/ElyesChouikha/DynamicKGConstruction.git", str(REPO_PATH)], check=True)
else:
    print("📦 Pulling latest...")
    subprocess.run(["git", "-C", str(REPO_PATH), "pull"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_PATH / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "langchain-anthropic", "neo4j", "pyshacl", "rdflib"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "numpy==1.26.4", "scipy", "scikit-learn", "--force-reinstall", "--no-cache-dir"], check=True)

print("✅ Setup complete")

## 1.b. Verify Imports

In [ ]:
import os, sys, getpass
from pathlib import Path

# Always resolve relative to the notebook file location
NOTEBOOK_DIR = Path(os.path.abspath("__file__" if "__file__" in dir() else ".")).parent
if not (NOTEBOOK_DIR / "forked_kg_construction").exists():
    NOTEBOOK_DIR = Path.cwd()

REPO_PATH = (NOTEBOOK_DIR / "forked_kg_construction").resolve()
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Repo path:    {REPO_PATH}")

sys.path.insert(0, str(REPO_PATH))
sys.path.insert(0, str(NOTEBOOK_DIR))
os.chdir(REPO_PATH)

import numpy as np
print(f"ℹ️ numpy version: {np.__version__}")
assert int(np.__version__.split(".")[0]) < 2, f"❌ numpy too new: {np.__version__}"

from skgb import SKGBConfig
from skgb.models import ModelRegistry
print("✅ SKGB imported successfully")

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")
key = os.environ["ANTHROPIC_API_KEY"]
print(f"✅ API key set: {key[:8]}...{key[-4:]}")

## 2. Start Ollama (embeddings only)

Ollama is only needed for `nomic-embed-text` embeddings (entity deduplication).
The LLM extraction goes to Claude via API — no large model download needed.

**If you have the Ollama app installed:** just make sure it's running before proceeding.
Skip the code cell below and go straight to **3.a Logging**.

**If running headless:** run the code cell below to start Ollama programmatically.

In [ ]:
import subprocess, time, urllib.request

# ── Check if Ollama is already running ───────────────────────────────────────
ollama_proc = None
try:
    urllib.request.urlopen("http://localhost:11434")
    print("✅ Ollama is already running (app or existing process)")
except Exception:
    print("⚠️ Ollama not running — starting it now...")
    ollama_proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(5)
    try:
        urllib.request.urlopen("http://localhost:11434")
        print(f"✅ Ollama started (PID {ollama_proc.pid})")
    except Exception as e:
        print(f"❌ Ollama still not responding: {e}")

# ── Pull embeddings model if not already present ─────────────────────────────
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
if "nomic-embed-text" in result.stdout:
    print("✅ nomic-embed-text already installed")
else:
    print("📥 Pulling nomic-embed-text (~274MB)...")
    subprocess.run(["ollama", "pull", "nomic-embed-text"])
    print("✅ Embeddings model ready")

## 3.a. Logging

Upload your own PDF or use the sample download below.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")
print("✅ Logging configured")

## 3.b. Upload PDF

In [ ]:
from pathlib import Path

INPUT_DIR = Path("../supply_chain_docs")
INPUT_DIR.mkdir(exist_ok=True)

pdfs = list(INPUT_DIR.glob("*.pdf"))
print(f"\n📂 {len(pdfs)} PDF(s) found\n")
for i, pdf in enumerate(pdfs):
    print(f"  [{i + 1}] {pdf.name}")

selected_pdfs = pdfs
print(f"\n✅ Selected {len(selected_pdfs)} file(s)")

## 4. Configure Pipeline

In [ ]:
from pathlib import Path
from skgb import SKGBConfig

cfg = SKGBConfig.from_out_dir(
    "skgb_output",
    llm_model="claude-sonnet-4-6",
    embeddings_model="nomic-embed-text",
    api_key=os.environ["ANTHROPIC_API_KEY"],
    ollama_base_url="http://localhost:11434",
    temperature=0.0,
    ent_threshold=0.8,
    rel_threshold=0.7,
    max_workers=2,
    min_chunk_words=200,
    max_chunk_words=800,
    overlap_words=0,
)

pdf_path = Path("../supply_chain_docs")
print(f"Processing all files from: {pdf_path}")

print(f"\nPipeline config:")
print(f"  LLM model:        {cfg.llm_model}")
print(f"  Embeddings model: {cfg.embeddings_model}")
print(f"  Ollama URL:       {cfg.ollama_base_url}")
print(f"  Output dir:       {cfg.out_dir}")

## 4.b Runtime Estimation (Optional)

In [ ]:
# ── Optional: Chunk estimate (runs Docling — takes ~1 min per file) ──────────
from skgb.adapters.docling_adapter import docling_convert_to_markdown
from skgb.adapters.chunking_adapter import chunk_markdown_files

print("📊 Estimating chunks for selected files...")
total_chunks = 0
for p in selected_pdfs:
    md_paths = docling_convert_to_markdown(
        input_path=p,
        output_dir=cfg.build_docling_dir,
        recursive=True,
    )
    chunks = chunk_markdown_files(
        md_paths=md_paths,
        min_chunk_words=cfg.min_chunk_words,
        max_chunk_words=cfg.max_chunk_words,
        overlap_words=cfg.overlap_words,
        preserve_metadata=cfg.preserve_metadata,
    )
    n = len(chunks)
    total_chunks += n
    print(f"   {p.name}: {n} chunks")

print(f"\n📦 Total chunks: {total_chunks}")
print(f"⏱️  Est. time: ~{total_chunks * 0.3:.0f} min (Claude API is ~3x faster than local model)")

## 5. Run the Pipeline

In [ ]:
import json
from pathlib import Path
from skgb.adapters.docling_adapter import docling_convert_to_markdown
from skgb.adapters.chunking_adapter import chunk_markdown_files

print("✅ Running pipeline...\n")

cfg.out_dir.mkdir(parents=True, exist_ok=True)
cfg.build_docling_dir.mkdir(parents=True, exist_ok=True)
cfg.chunks_output_dir.mkdir(parents=True, exist_ok=True)

print("📄 Converting all PDFs to Markdown...")
md_paths = docling_convert_to_markdown(
    input_path=Path("../supply_chain_docs"),
    output_dir=cfg.build_docling_dir,
    recursive=False,
)
print(f"✅ Converted {len(md_paths)} file(s)\n")

print("✂️  Chunking documents...")
all_chunks = chunk_markdown_files(
    md_paths=md_paths,
    min_chunk_words=cfg.min_chunk_words,
    max_chunk_words=cfg.max_chunk_words,
    overlap_words=cfg.overlap_words,
    preserve_metadata=cfg.preserve_metadata,
)
print(f"✅ {len(all_chunks)} chunks generated\n")

chunks_json_path = cfg.chunks_output_dir / "all_chunks.json"
chunks_json_path.write_text(
    json.dumps(all_chunks, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"✅ {len(all_chunks)} total chunks saved to {chunks_json_path}")

## 6. Chunks Preview

In [ ]:
import json
from pathlib import Path

chunks_json_path = Path("skgb_output/chunks_output/all_chunks.json")
assert chunks_json_path.exists(), "❌ Run the Docling cell first"

all_chunks = json.loads(chunks_json_path.read_text())
print(f"✅ {len(all_chunks)} chunks loaded\n")

for i, ch in enumerate(all_chunks[:3]):
    print(f"--- Chunk {i+1} ---")
    print(f"  Section: {ch.get('section_title', 'N/A')}")
    content = ch.get('content', '')
    print(f"  Content: {content[:200]}{'...' if len(content) > 200 else ''}")
    print()

## 7. Ontology-Guided Extraction

Runs **on top of** the chunks already produced by the pipeline above.
Extracts entities mapped directly to `resilience_ontology.ttl` classes:
`Supplier`, `Facility`, `Location`, `Product`, `RiskEvent`, `Certification`.

In [ ]:
import os, json
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_core.prompts import ChatPromptTemplate
from skgb.models import ModelRegistry

# ── Pydantic schema mirroring resilience_ontology.ttl ────────────────────────

class Location(BaseModel):
    countryName: Optional[str] = Field(None, description="Country, e.g. India, Vietnam")
    regionName: Optional[str] = Field(None, description="Region, state, or city")

class Certification(BaseModel):
    certificationName: Optional[str] = Field(None, description="e.g. ISO 22301")
    issueDate: Optional[str] = Field(None, description="Date issued if mentioned")

class RiskEvent(BaseModel):
    eventName: str = Field(description="Name or description of the disruption")
    revenueImpact: Optional[float] = Field(None, description="Financial impact in USD")
    timeToAwareness: Optional[int] = Field(None, description="Days to detect the issue")
    timeToAction: Optional[int] = Field(None, description="Days between awareness and response")
    affectsCountry: Optional[str] = Field(None, description="Country where event occurs")

class Facility(BaseModel):
    facilityName: str = Field(description="Name of the manufacturing facility or plant")
    timeToRecover: Optional[int] = Field(None, description="TTR in days")
    timeToSurvive: Optional[int] = Field(None, description="TTS in days")
    isDualSourced: Optional[bool] = Field(None, description="True if dual sourcing is mentioned")
    inventoryVelocity: Optional[float] = Field(None, description="Inventory turnover rate")
    adaptabilityIndex: Optional[str] = Field(None, description="High/Medium/Low or qualitative")
    digitalizationLevel: Optional[str] = Field(None, description="Level of digital/AI integration")
    locatedIn: Optional[Location] = Field(None, description="Physical location")

class Supplier(BaseModel):
    supplierName: str = Field(description="Corporate entity name, e.g. Foxconn, Tata Electronics")
    tierLevel: Optional[str] = Field(None, description="Tier 1, Tier 2, OEM etc.")
    riskIndex: Optional[float] = Field(None, description="Supplier risk score 0.0-1.0")
    collaborationLevel: Optional[str] = Field(None, description="Strategic/Integrated/Standard")
    sharesData: Optional[bool] = Field(None, description="True if shares real-time data with partners")
    certifications: List[Certification] = Field(default_factory=list)
    owns_facilities: List[Facility] = Field(default_factory=list)

class Product(BaseModel):
    productName: str = Field(description="Product name, e.g. PCB, smartphone, EV chassis")
    leadTime: Optional[float] = Field(None, description="Lead time in days")
    onTimeDeliveryRate: Optional[float] = Field(None, description="On-time delivery rate %")
    isDualSourced: Optional[bool] = Field(None, description="True if dual sourced")

class SupplyChainKG(BaseModel):
    """Top-level KG output — mirrors resilience_ontology.ttl"""
    suppliers: List[Supplier] = Field(default_factory=list)
    products: List[Product] = Field(default_factory=list)
    risk_events: List[RiskEvent] = Field(default_factory=list)

# ── Init Claude ───────────────────────────────────────────────────────────────

llm = ModelRegistry.create_llm(
    "claude-sonnet-4-6",
    temperature=0.0,
    api_key=os.environ["ANTHROPIC_API_KEY"],
)
structured_llm = llm.with_structured_output(SupplyChainKG)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a Supply Chain Resilience Analyst extracting structured data for a knowledge graph.\n"
     "Classes: Supplier, Facility, Location, Product, RiskEvent, Certification.\n"
     "Rules:\n"
     "- A Supplier OWNS one or more Facilities.\n"
     "- A Facility is LOCATED_IN exactly one Location.\n"
     "- Extract TTR and TTS in DAYS only — convert weeks/months if needed.\n"
     "- isDualSourced = True only if dual sourcing is explicitly mentioned.\n"
     "- If a metric is not in the text, leave it null. Never invent numbers.\n"
     "- Focus on India and Vietnam operations.\n"
     "- A Supplier must be a manufacturing company that owns physical production facilities.\n"
     "- Do NOT classify financial institutions, patent holders, regulators, stock exchanges, pension funds, or customers as Suppliers.\n"
     "- For each Facility, explicitly link it to the products it manufactures using the owns_facilities list.\n"
     "- For each RiskEvent, set affectsCountry to the country most affected (e.g. India, Vietnam, Taiwan).\n"
     "- If the document context makes clear which company is being described (e.g. the document title or header identifies the company), use that company name even if not explicitly stated in the chunk.\n"
     "- Extract ALL named facilities explicitly mentioned in the text, including planned or under-construction ones.\n"
     "- Opportunities and positive market trends are NOT RiskEvents. Only extract disruptions, threats, and negative events as RiskEvents."),
    ("human", "{text}")
])
extraction_chain = prompt | structured_llm

# ── Load chunks (flat list — each has a 'content' key) ───────────────────────

with open('skgb_output/chunks_output/all_chunks.json', 'r') as f:
    all_chunks = json.load(f)

print(f"Total chunks loaded: {len(all_chunks)}")
print(f"First chunk — doc:     {all_chunks[0].get('metadata', {}).get('doc_name', 'N/A')}")
print(f"First chunk — section: {all_chunks[0].get('section_title', 'N/A')}")

# ── Test on first chunk ───────────────────────────────────────────────────────

test_chunk = all_chunks[0]["content"]
print(f"\nExtracting from chunk ({len(test_chunk.split())} words)...\n")

result_kg = extraction_chain.invoke({"text": test_chunk})
print("─── KG OUTPUT ───")
print(result_kg.model_dump_json(indent=2))

In [ ]:
import time
from pathlib import Path

# ── Run extraction over all chunks ───────────────────────────────────────────
all_suppliers = []
all_products = []
all_risk_events = []
failed_chunks = []
total = len(all_chunks)

print(f"Processing {total} chunks with Claude...\n")

for i, chunk in enumerate(all_chunks):
    text = chunk.get("content", "")
    section = chunk.get("section_title", "N/A")
    doc = chunk.get("metadata", {}).get("doc_name", "N/A")

    # Skip very short chunks — not enough context for extraction
    if len(text.split()) < 30:
        print(f"  [{i+1}/{total}] Skipping short chunk ({len(text.split())} words)")
        continue

    try:
        result_kg = extraction_chain.invoke({"text": text})
        all_suppliers.extend(result_kg.suppliers)
        all_products.extend(result_kg.products)
        all_risk_events.extend(result_kg.risk_events)

        extracted = len(result_kg.suppliers) + len(result_kg.products) + len(result_kg.risk_events)
        print(f"  [{i+1}/{total}] ✅ {doc} | {section[:40]} → {extracted} entities")

    except Exception as e:
        print(f"  [{i+1}/{total}] ❌ Failed: {e}")
        failed_chunks.append({"index": i, "section": section, "error": str(e)})

    # Save checkpoint every 100 chunks
    if (i + 1) % 100 == 0:
        checkpoint = {
            "suppliers": [s.model_dump() for s in all_suppliers],
            "products": [p.model_dump() for p in all_products],
            "risk_events": [r.model_dump() for r in all_risk_events],
            "last_chunk_index": i
        }
        Path("skgb_output/checkpoint_partial.json").write_text(
            json.dumps(checkpoint, indent=2, ensure_ascii=False)
        )
        print(f"  💾 Checkpoint saved at chunk {i+1}")

    # Respect Anthropic rate limits — ~1 request/sec is safe on Sonnet
    time.sleep(1.0)

print(f"\n{'='*60}")
print(f"Extraction complete!")
print(f"  Suppliers extracted:   {len(all_suppliers)}")
print(f"  Products extracted:    {len(all_products)}")
print(f"  Risk events extracted: {len(all_risk_events)}")
print(f"  Failed chunks:         {len(failed_chunks)}")

In [ ]:
# ── Deduplicate by name and save to JSON ─────────────────────────────────────
def dedup_by_name(items, name_field):
    seen = {}
    for item in items:
        key = getattr(item, name_field, "").strip().lower()
        if not key:
            continue
        if key not in seen:
            seen[key] = item
        else:
            existing = seen[key].model_dump()
            new = item.model_dump()
            merged = {k: (existing[k] if existing[k] is not None else new[k])
                      for k in existing}
            seen[key] = item.__class__(**merged)
    return list(seen.values())

deduped_suppliers = dedup_by_name(all_suppliers, "supplierName")
deduped_products = dedup_by_name(all_products, "productName")
deduped_risk_events = dedup_by_name(all_risk_events, "eventName")

print(f"After deduplication:")
print(f"  Suppliers:   {len(all_suppliers)} → {len(deduped_suppliers)}")
print(f"  Products:    {len(all_products)} → {len(deduped_products)}")
print(f"  Risk events: {len(all_risk_events)} → {len(deduped_risk_events)}")

# Save to disk
output = {
    "suppliers": [s.model_dump() for s in deduped_suppliers],
    "products": [p.model_dump() for p in deduped_products],
    "risk_events": [r.model_dump() for r in deduped_risk_events],
    "extraction_stats": {
        "total_chunks": total,
        "failed_chunks": len(failed_chunks),
        "failed_details": failed_chunks
    }
}

out_path = Path("skgb_output/ontology_extraction.json")
out_path.write_text(json.dumps(output, indent=2, ensure_ascii=False))
print(f"\n✅ Saved to {out_path}")

# Preview
print(f"\n--- First 3 Suppliers ---")
for s in deduped_suppliers[:3]:
    print(f"  {s.supplierName} | Tier: {s.tierLevel} | Risk: {s.riskIndex}")
    for f in s.owns_facilities:
        print(f"    └─ {f.facilityName} | TTR: {f.timeToRecover}d | Location: {f.locatedIn}")

print(f"\n--- First 3 Products ---")
for p in deduped_products[:3]:
    print(f"  {p.productName} | Lead time: {p.leadTime}d | OTD: {p.onTimeDeliveryRate}%")

print(f"\n--- First 3 Risk Events ---")
for r in deduped_risk_events[:3]:
    print(f"  {r.eventName} | Country: {r.affectsCountry} | Impact: ${r.revenueImpact}")

## 8. Neo4j Loading

In [ ]:
from neo4j import GraphDatabase
import json, getpass
from pathlib import Path

# ── Load extraction results ───────────────────────────────────────────────────
out_path = Path("skgb_output/ontology_extraction.json")
assert out_path.exists(), "❌ Run the extraction cell first"

data = json.loads(out_path.read_text())
suppliers = data["suppliers"]
products = data["products"]
risk_events = data["risk_events"]

print(f"Loaded: {len(suppliers)} suppliers, {len(products)} products, {len(risk_events)} risk events")

# ── Connect to Neo4j (single connection for entire cell) ─────────────────────
NEO4J_URI = "neo4j+s://7973b943.databases.neo4j.io"
NEO4J_USER = getpass.getpass("Neo4j username: ")
NEO4J_PASSWORD = getpass.getpass("Neo4j password: ")
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_query(tx, query, **params):
    tx.run(query, **params)

# ── Clear existing graph ──────────────────────────────────────────────────────
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
    print("✅ Graph cleared")

# ── Load Suppliers and Facilities ─────────────────────────────────────────────
with driver.session() as session:
    for s in suppliers:
        session.execute_write(run_query, """
            MERGE (sup:Supplier {name: $name})
            SET sup.tierLevel = $tierLevel,
                sup.riskIndex = $riskIndex,
                sup.collaborationLevel = $collaborationLevel,
                sup.sharesData = $sharesData
        """,
            name=s["supplierName"],
            tierLevel=s.get("tierLevel"),
            riskIndex=s.get("riskIndex"),
            collaborationLevel=s.get("collaborationLevel"),
            sharesData=s.get("sharesData"),
        )

        for cert in s.get("certifications", []):
            session.execute_write(run_query, """
                MATCH (sup:Supplier {name: $supplierName})
                MERGE (c:Certification {name: $certName})
                SET c.issueDate = $issueDate
                MERGE (sup)-[:HAS_CERTIFICATION]->(c)
            """,
                supplierName=s["supplierName"],
                certName=cert.get("certificationName"),
                issueDate=cert.get("issueDate"),
            )

        for fac in s.get("owns_facilities", []):
            loc = fac.get("locatedIn") or {}
            country = loc.get("countryName")
            region = loc.get("regionName")

            session.execute_write(run_query, """
                MATCH (sup:Supplier {name: $supplierName})
                MERGE (f:Facility {name: $facName})
                SET f.timeToRecover = $ttr,
                    f.timeToSurvive = $tts,
                    f.isDualSourced = $dual,
                    f.inventoryVelocity = $inv,
                    f.adaptabilityIndex = $adapt,
                    f.digitalizationLevel = $digital
                MERGE (sup)-[:OWNS]->(f)
            """,
                supplierName=s["supplierName"],
                facName=fac.get("facilityName"),
                ttr=fac.get("timeToRecover"),
                tts=fac.get("timeToSurvive"),
                dual=fac.get("isDualSourced"),
                inv=fac.get("inventoryVelocity"),
                adapt=fac.get("adaptabilityIndex"),
                digital=fac.get("digitalizationLevel"),
            )

            if country:
                session.execute_write(run_query, """
                    MATCH (f:Facility {name: $facName})
                    MERGE (l:Location {country: $country})
                    SET l.region = $region
                    MERGE (f)-[:LOCATED_IN]->(l)
                """,
                    facName=fac.get("facilityName"),
                    country=country,
                    region=region,
                )

    print(f"✅ Loaded {len(suppliers)} suppliers with facilities")

# ── Load Products ─────────────────────────────────────────────────────────────
with driver.session() as session:
    for p in products:
        session.execute_write(run_query, """
            MERGE (prod:Product {name: $name})
            SET prod.leadTime = $leadTime,
                prod.onTimeDeliveryRate = $otd,
                prod.isDualSourced = $dual
        """,
            name=p["productName"],
            leadTime=p.get("leadTime"),
            otd=p.get("onTimeDeliveryRate"),
            dual=p.get("isDualSourced"),
        )
    print(f"✅ Loaded {len(products)} products")

# ── Load Risk Events ──────────────────────────────────────────────────────────
with driver.session() as session:
    for r in risk_events:
        session.execute_write(run_query, """
            MERGE (e:RiskEvent {name: $name})
            SET e.revenueImpact = $impact,
                e.timeToAwareness = $tta,
                e.timeToAction = $ttac,
                e.affectsCountry = $country
        """,
            name=r["eventName"],
            impact=r.get("revenueImpact"),
            tta=r.get("timeToAwareness"),
            ttac=r.get("timeToAction"),
            country=r.get("affectsCountry"),
        )
    print(f"✅ Loaded {len(risk_events)} risk events")

# ── Link RiskEvents to Locations via AFFECTS ─────────────────────────────────
with driver.session() as session:
    count = 0
    for r in risk_events:
        if r.get("affectsCountry"):
            session.execute_write(run_query, """
                MATCH (e:RiskEvent {name: $eventName})
                MERGE (l:Location {country: $country})
                MERGE (e)-[:AFFECTS]->(l)
            """,
                eventName=r["eventName"],
                country=r["affectsCountry"],
            )
            count += 1
    print(f"✅ Loaded {count} AFFECTS relationships")

# ── Link Facilities to Products via MANUFACTURES (bulk) ──────────────────────
with driver.session() as session:
    session.run("""
        MATCH (sup:Supplier)-[:OWNS]->(f:Facility)
        MATCH (prod:Product)
        MERGE (f)-[:MANUFACTURES]->(prod)
    """)
    result = session.run("""
        MATCH ()-[r:MANUFACTURES]->() RETURN count(r) as count
    """)
    count = result.single()["count"]
    print(f"✅ Loaded {count} MANUFACTURES relationships")

# ── Link RiskEvents to Facilities via DISRUPTS ────────────────────────────────
with driver.session() as session:
    count = 0
    for r in risk_events:
        country = r.get("affectsCountry")
        if country:
            session.execute_write(run_query, """
                MATCH (e:RiskEvent {name: $eventName})
                MATCH (f:Facility)-[:LOCATED_IN]->(l:Location {country: $country})
                MERGE (e)-[:DISRUPTS]->(f)
            """,
                eventName=r["eventName"],
                country=country,
            )
            count += 1
    print(f"✅ Loaded {count} DISRUPTS relationships")

# ── Final summary ─────────────────────────────────────────────────────────────
with driver.session() as session:
    result = session.run("MATCH (n) RETURN labels(n)[0] as label, count(n) as count")
    print(f"\n📊 Graph summary:")
    for record in result:
        print(f"  {record['label']:20s} {record['count']} nodes")

    result = session.run("MATCH ()-[r]->() RETURN type(r) as type, count(r) as count")
    print(f"\n🔗 Relationships:")
    for record in result:
        print(f"  {record['type']:20s} {record['count']} edges")

driver.close()
print("\n✅ Neo4j loading complete")

## 9. SHACL Validation

In [ ]:
from pyshacl import validate
from rdflib import Graph, Namespace, RDF, Literal, XSD
from pathlib import Path

# ── Build RDF graph from extracted data ──────────────────────────────────────
import json

data = json.loads(Path("skgb_output/ontology_extraction.json").read_text())

NS = Namespace("http://example.org/supply-chain-resilience#")
g = Graph()
g.bind("", NS)

def uri(name):
    # Clean name for use as URI — replace spaces with underscores
    return NS[name.replace(" ", "_").replace("/", "_")]

# Load Suppliers
for s in data["suppliers"]:
    sup_uri = uri(s["supplierName"])
    g.add((sup_uri, RDF.type, NS.Supplier))

    for fac in s.get("owns_facilities", []):
        fac_uri = uri(fac["facilityName"])
        g.add((fac_uri, RDF.type, NS.Facility))
        g.add((sup_uri, NS.owns, fac_uri))

        if fac.get("timeToRecover") is not None:
            g.add((fac_uri, NS.timeToRecover, Literal(fac["timeToRecover"], datatype=XSD.integer)))

        loc = fac.get("locatedIn") or {}
        if loc.get("countryName"):
            loc_uri = uri(loc["countryName"])
            g.add((loc_uri, RDF.type, NS.Location))
            g.add((loc_uri, NS.countryName, Literal(loc["countryName"])))
            if loc.get("regionName"):
                g.add((loc_uri, NS.regionName, Literal(loc["regionName"])))
            g.add((fac_uri, NS.locatedIn, loc_uri))

# Load Risk Events
for r in data["risk_events"]:
    evt_uri = uri(r["eventName"])
    g.add((evt_uri, RDF.type, NS.RiskEvent))
    if r.get("revenueImpact") is not None:
        g.add((evt_uri, NS.revenueImpact, Literal(r["revenueImpact"], datatype=XSD.float)))

print(f"✅ RDF graph built: {len(g)} triples")

# ── Load SHACL shapes ─────────────────────────────────────────────────────────
shapes_path = Path("ontology/resilience_shapes.ttl")
assert shapes_path.exists(), "❌ Upload resilience_shapes.ttl to /content first"

# ── Run SHACL validation ──────────────────────────────────────────────────────
conforms, results_graph, results_text = validate(
    data_graph=g,
    shacl_graph=str(shapes_path),
    data_graph_format="turtle",
    shacl_graph_format="turtle",
    inference="rdfs",
    debug=False,
)

print(f"\n{'='*60}")
print(f"SHACL Validation {'PASSED ✅' if conforms else 'FAILED ❌'}")
print(f"{'='*60}\n")

# ── Parse and count results ───────────────────────────────────────────────────
SH = Namespace("http://www.w3.org/ns/shacl#")
errors = []
warnings = []

for result in results_graph.subjects(RDF.type, SH.ValidationResult):
    severity = results_graph.value(result, SH.resultSeverity)
    message = results_graph.value(result, SH.resultMessage)
    focus = results_graph.value(result, SH.focusNode)
    path = results_graph.value(result, SH.resultPath)

    entry = {
        "node": str(focus),
        "path": str(path),
        "message": str(message),
    }

    if severity == SH.Warning:
        warnings.append(entry)
    else:
        errors.append(entry)

total_nodes = len(set(g.subjects(RDF.type, None)))
total_violations = len(errors)
pass_rate = ((total_nodes - total_violations) / total_nodes * 100) if total_nodes > 0 else 0

print(f"📊 Validation Summary:")
print(f"  Total nodes:      {total_nodes}")
print(f"  Errors:           {len(errors)}")
print(f"  Warnings:         {len(warnings)}")
print(f"  SHACL pass rate:  {pass_rate:.1f}%  (target ≥ 85%)")
print(f"  {'✅ Target met' if pass_rate >= 85 else '❌ Below target'}\n")

if errors:
    print("── Errors ──────────────────────────────────────────────")
    for e in errors[:10]:
        print(f"  Node:    {e['node'].split('#')[-1]}")
        print(f"  Path:    {e['path'].split('#')[-1]}")
        print(f"  Message: {e['message']}")
        print()

if warnings:
    print("── Warnings ─────────────────────────────────────────────")
    for w in warnings[:10]:
        print(f"  Node:    {w['node'].split('#')[-1]}")
        print(f"  Message: {w['message']}")
        print()

# Save validation report
report = {
    "conforms": conforms,
    "pass_rate": round(pass_rate, 2),
    "total_nodes": total_nodes,
    "error_count": len(errors),
    "warning_count": len(warnings),
    "errors": errors,
    "warnings": warnings,
}
report_path = Path("skgb_output/shacl_report.json")
report_path.write_text(json.dumps(report, indent=2))
print(f"✅ SHACL report saved to {report_path}")

## 10. CQ Answerability

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

CQS = [
    # ── Visibility ────────────────────────────────────────────────────────────
    {
        "id": "CQ1",
        "question": "What is the average reported Time to Awareness for supply chain disruptions among Tier 1 electronics suppliers in Vietnam?",
        "cypher": """
            MATCH (e:RiskEvent)-[:AFFECTS]->(l:Location)
            WHERE l.country = 'Vietnam'
            AND e.timeToAwareness IS NOT NULL
            RETURN avg(e.timeToAwareness) AS avg_time_to_awareness_days
        """
    },
    {
        "id": "CQ2",
        "question": "Compare the standard Order Lead Time for PCBs sourced from India versus Vietnam in 2024.",
        "cypher": """
            MATCH (f:Facility)-[:LOCATED_IN]->(l:Location)
            MATCH (f)-[:MANUFACTURES]->(p:Product)
            WHERE p.name CONTAINS 'PCB'
            AND l.country IN ['India', 'Vietnam']
            AND p.leadTime IS NOT NULL
            RETURN l.country AS country, avg(p.leadTime) AS avg_lead_time_days
            ORDER BY country
        """
    },
    {
        "id": "CQ3",
        "question": "Which suppliers in India have reported an On-time Delivery Rate of less than 95%?",
        "cypher": """
            MATCH (s:Supplier)-[:OWNS]->(f:Facility)-[:LOCATED_IN]->(l:Location)
            MATCH (f)-[:MANUFACTURES]->(p:Product)
            WHERE l.country = 'India'
            AND p.onTimeDeliveryRate IS NOT NULL
            AND p.onTimeDeliveryRate < 95
            RETURN s.name AS supplier, p.name AS product, p.onTimeDeliveryRate AS otd_rate
            ORDER BY otd_rate
        """
    },
    {
        "id": "CQ4",
        "question": "List disruption events in Vietnam where Time to Action was greater than 48 hours.",
        "cypher": """
            MATCH (e:RiskEvent)-[:AFFECTS]->(l:Location)
            WHERE l.country = 'Vietnam'
            AND e.timeToAction IS NOT NULL
            AND e.timeToAction > 2
            RETURN e.name AS event, e.timeToAction AS time_to_action_days
            ORDER BY time_to_action_days DESC
        """
    },
    # ── Flexibility ───────────────────────────────────────────────────────────
    {
        "id": "CQ5",
        "question": "Which semiconductor assembly facilities in India have a TTR of less than two weeks?",
        "cypher": """
            MATCH (s:Supplier)-[:OWNS]->(f:Facility)-[:LOCATED_IN]->(l:Location)
            WHERE l.country = 'India'
            AND f.timeToRecover IS NOT NULL
            AND f.timeToRecover < 14
            RETURN s.name AS supplier, f.name AS facility, f.timeToRecover AS ttr_days
            ORDER BY ttr_days
        """
    },
    {
        "id": "CQ6",
        "question": "How has Inventory Velocity changed for major electronics firms shifting production from China to Vietnam between 2022 and 2025?",
        "cypher": """
            MATCH (s:Supplier)-[:OWNS]->(f:Facility)-[:LOCATED_IN]->(l:Location)
            WHERE l.country = 'Vietnam'
            AND f.inventoryVelocity IS NOT NULL
            RETURN s.name AS supplier, f.name AS facility, f.inventoryVelocity AS inventory_velocity
            ORDER BY inventory_velocity DESC
        """
    },
    {
        "id": "CQ7",
        "question": "Which firms explicitly mention having a high Adaptability Index in their Indian manufacturing hubs?",
        "cypher": """
            MATCH (s:Supplier)-[:OWNS]->(f:Facility)-[:LOCATED_IN]->(l:Location)
            WHERE l.country = 'India'
            AND f.adaptabilityIndex IS NOT NULL
            RETURN s.name AS supplier, f.name AS facility, f.adaptabilityIndex AS adaptability
            ORDER BY supplier
        """
    },
    {
        "id": "CQ8",
        "question": "List facilities in Vietnam that can switch production between different component types within 24 hours.",
        "cypher": """
            MATCH (s:Supplier)-[:OWNS]->(f:Facility)-[:LOCATED_IN]->(l:Location)
            WHERE l.country = 'Vietnam'
            AND f.adaptabilityIndex IS NOT NULL
            RETURN s.name AS supplier, f.name AS facility, f.adaptabilityIndex AS adaptability
            ORDER BY supplier
        """
    },
    # ── Diversification ───────────────────────────────────────────────────────
    {
        "id": "CQ9",
        "question": "Which Tier 1 suppliers in India have a TTS of less than 7 days?",
        "cypher": """
            MATCH (s:Supplier)-[:OWNS]->(f:Facility)-[:LOCATED_IN]->(l:Location)
            WHERE l.country = 'India'
            AND s.tierLevel = 'Tier 1'
            AND f.timeToSurvive IS NOT NULL
            AND f.timeToSurvive < 7
            RETURN s.name AS supplier, f.name AS facility, f.timeToSurvive AS tts_days
            ORDER BY tts_days
        """
    },
    {
        "id": "CQ10",
        "question": "List electronics suppliers in Vietnam classified with a High Supplier Risk Index.",
        "cypher": """
            MATCH (s:Supplier)-[:OWNS]->(f:Facility)-[:LOCATED_IN]->(l:Location)
            WHERE l.country = 'Vietnam'
            AND s.riskIndex IS NOT NULL
            AND s.riskIndex >= 0.7
            RETURN s.name AS supplier, s.riskIndex AS risk_index
            ORDER BY risk_index DESC
        """
    },
    {
        "id": "CQ11",
        "question": "Which firms have explicitly reported incurring Resilience Costs in their India operations?",
        "cypher": """
            MATCH (e:RiskEvent)-[:DISRUPTS]->(f:Facility)-[:LOCATED_IN]->(l:Location)
            WHERE l.country = 'India'
            AND e.revenueImpact IS NOT NULL
            MATCH (s:Supplier)-[:OWNS]->(f)
            RETURN s.name AS supplier, e.name AS event, e.revenueImpact AS revenue_impact_usd
            ORDER BY revenue_impact_usd DESC
        """
    },
    {
        "id": "CQ12",
        "question": "What is the Recovery Rate for key Vietnamese industrial parks following a major power outage?",
        "cypher": """
            MATCH (e:RiskEvent)-[:DISRUPTS]->(f:Facility)-[:LOCATED_IN]->(l:Location)
            WHERE l.country = 'Vietnam'
            AND f.timeToRecover IS NOT NULL
            RETURN f.name AS facility, f.timeToRecover AS ttr_days,
                   round(100.0 / f.timeToRecover, 2) AS recovery_rate_pct_per_day
            ORDER BY recovery_rate_pct_per_day DESC
        """
    },
    {
        "id": "CQ13",
        "question": "Which critical components are currently dual-sourced from both India and Vietnam?",
        "cypher": """
            MATCH (p:Product)
            WHERE p.isDualSourced = true
            MATCH (f:Facility)-[:MANUFACTURES]->(p)
            MATCH (f)-[:LOCATED_IN]->(l:Location)
            WHERE l.country IN ['India', 'Vietnam']
            RETURN p.name AS product, collect(DISTINCT l.country) AS sourced_from
            ORDER BY product
        """
    },
    # ── Collaboration ─────────────────────────────────────────────────────────
    {
        "id": "CQ14",
        "question": "Which suppliers in India are mentioned as having strategic Collaboration Effectiveness with major OEMs?",
        "cypher": """
            MATCH (s:Supplier)-[:OWNS]->(f:Facility)-[:LOCATED_IN]->(l:Location)
            WHERE l.country = 'India'
            AND s.collaborationLevel IS NOT NULL
            AND s.collaborationLevel IN ['Strategic', 'strategic', 'deeply integrated']
            RETURN s.name AS supplier, s.collaborationLevel AS collaboration_level
            ORDER BY supplier
        """
    },
    {
        "id": "CQ15",
        "question": "Compare Supplier Delivery Efficiency for logistics partners in Northern vs Southern Vietnam.",
        "cypher": """
            MATCH (s:Supplier)-[:OWNS]->(f:Facility)-[:LOCATED_IN]->(l:Location)
            WHERE l.country = 'Vietnam'
            AND l.region IS NOT NULL
            RETURN l.region AS region, s.name AS supplier,
                   avg(s.riskIndex) AS avg_risk_index
            ORDER BY region, avg_risk_index
        """
    },
    {
        "id": "CQ16",
        "question": "List electronics firms that share real-time inventory data with their Tier 2 suppliers in India.",
        "cypher": """
            MATCH (s:Supplier)-[:OWNS]->(f:Facility)-[:LOCATED_IN]->(l:Location)
            WHERE l.country = 'India'
            AND s.sharesData = true
            RETURN s.name AS supplier, s.tierLevel AS tier, s.sharesData AS shares_real_time_data
            ORDER BY supplier
        """
    },
    # ── Technology & General ──────────────────────────────────────────────────
    {
        "id": "CQ17",
        "question": "Which manufacturing facilities in Vietnam have a high Digitalization Level?",
        "cypher": """
            MATCH (s:Supplier)-[:OWNS]->(f:Facility)-[:LOCATED_IN]->(l:Location)
            WHERE l.country = 'Vietnam'
            AND f.digitalizationLevel IS NOT NULL
            RETURN s.name AS supplier, f.name AS facility, f.digitalizationLevel AS digitalization
            ORDER BY supplier
        """
    },
    {
        "id": "CQ18",
        "question": "What was the reported financial impact for electronics firms due to supply chain disruptions in India during the last fiscal year?",
        "cypher": """
            MATCH (e:RiskEvent)-[:DISRUPTS]->(f:Facility)-[:LOCATED_IN]->(l:Location)
            WHERE l.country = 'India'
            AND e.revenueImpact IS NOT NULL
            MATCH (s:Supplier)-[:OWNS]->(f)
            RETURN s.name AS supplier, sum(e.revenueImpact) AS total_revenue_impact_usd
            ORDER BY total_revenue_impact_usd DESC
        """
    },
    {
        "id": "CQ19",
        "question": "List companies that have cited an improved Resilience Value after expanding into Vietnam.",
        "cypher": """
            MATCH (s:Supplier)-[:OWNS]->(f:Facility)-[:LOCATED_IN]->(l:Location)
            WHERE l.country = 'Vietnam'
            AND f.timeToRecover IS NOT NULL
            AND s.riskIndex IS NOT NULL
            RETURN s.name AS supplier,
                   f.timeToRecover AS ttr_days,
                   s.riskIndex AS risk_index
            ORDER BY ttr_days, risk_index
        """
    },
    {
        "id": "CQ20",
        "question": "Which suppliers in India possess ISO 22301 certification?",
        "cypher": """
            MATCH (s:Supplier)-[:HAS_CERTIFICATION]->(c:Certification)
            MATCH (s)-[:OWNS]->(f:Facility)-[:LOCATED_IN]->(l:Location)
            WHERE l.country = 'India'
            AND c.name CONTAINS '22301'
            RETURN s.name AS supplier, c.name AS certification, c.issueDate AS issue_date
            ORDER BY supplier
        """
    },
]

# ── Run all CQs ───────────────────────────────────────────────────────────────
results_summary = []

print(f"Running {len(CQS)} Competency Questions...\n")
print("=" * 60)

with driver.session() as session:
    for cq in CQS:
        try:
            records = session.run(cq["cypher"]).data()
            answerable = len(records) > 0
            results_summary.append({
                "id": cq["id"],
                "question": cq["question"],
                "answerable": answerable,
                "result_count": len(records),
                "sample": records[:2] if records else []
            })
            status = "✅" if answerable else "❌"
            print(f"{status} {cq['id']}: {cq['question'][:70]}...")
            if records:
                print(f"   → {len(records)} result(s): {records[0]}")
            print()
        except Exception as e:
            results_summary.append({
                "id": cq["id"],
                "question": cq["question"],
                "answerable": False,
                "error": str(e)
            })
            print(f"💥 {cq['id']}: Query error — {e}\n")

driver.close()

# ── Summary ───────────────────────────────────────────────────────────────────
answered = sum(1 for r in results_summary if r["answerable"])
answer_rate = answered / len(CQS) * 100

print("=" * 60)
print(f"📊 CQ Answerability Summary:")
print(f"  Answered:     {answered}/{len(CQS)}")
print(f"  Answer rate:  {answer_rate:.1f}%  (target ≥ 80%)")
print(f"  {'✅ Target met' if answer_rate >= 80 else '❌ Below target'}")

# Save report
cq_report_path = Path("skgb_output/cq_answerability.json")
cq_report_path.write_text(json.dumps(results_summary, indent=2))
print(f"\n✅ CQ report saved to {cq_report_path}")

## 11. Explore Results

In [ ]:
import json
import pandas as pd
import networkx as nx
from pathlib import Path
from IPython.display import display

# ── Load extraction results ───────────────────────────────────────────────────
data = json.loads(Path("skgb_output/ontology_extraction.json").read_text())
shacl = json.loads(Path("skgb_output/shacl_report.json").read_text())
cq = json.loads(Path("skgb_output/cq_answerability.json").read_text())

suppliers = data["suppliers"]
products = data["products"]
risk_events = data["risk_events"]
stats = data["extraction_stats"]

# ── Extraction summary ────────────────────────────────────────────────────────
print("=" * 60)
print("📊 Extraction Summary")
print("=" * 60)
print(f"  Total chunks processed: {stats['total_chunks']}")
print(f"  Failed chunks:          {stats['failed_chunks']}")
print(f"  Suppliers extracted:    {len(suppliers)}")
print(f"  Products extracted:     {len(products)}")
print(f"  Risk events extracted:  {len(risk_events)}")

total_facilities = sum(len(s.get("owns_facilities", [])) for s in suppliers)
total_certs = sum(len(s.get("certifications", [])) for s in suppliers)
print(f"  Facilities extracted:   {total_facilities}")
print(f"  Certifications:         {total_certs}")

# ── SHACL summary ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("🔍 SHACL Validation")
print("=" * 60)
print(f"  Pass rate:   {shacl['pass_rate']}%  (target ≥ 85%)")
print(f"  Errors:      {shacl['error_count']}")
print(f"  Warnings:    {shacl['warning_count']}")
print(f"  {'✅ Target met' if shacl['pass_rate'] >= 85 else '❌ Below target'}")

# ── CQ summary ────────────────────────────────────────────────────────────────
answered = sum(1 for r in cq if r["answerable"])
answer_rate = answered / len(cq) * 100
print(f"\n{'='*60}")
print("❓ CQ Answerability")
print("=" * 60)
print(f"  Answered:    {answered}/{len(cq)}")
print(f"  Rate:        {answer_rate:.1f}%  (target ≥ 80%)")
print(f"  {'✅ Target met' if answer_rate >= 80 else '❌ Below target'}")

unanswered = [r for r in cq if not r["answerable"]]
if unanswered:
    print(f"\n  Unanswered CQs:")
    for r in unanswered:
        print(f"    ❌ {r['id']}: {r['question'][:70]}...")

# ── Suppliers table ───────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("🏭 Suppliers")
print("=" * 60)
rows = []
for s in suppliers:
    for f in s.get("owns_facilities", []):
        loc = f.get("locatedIn") or {}
        rows.append({
            "Supplier": s["supplierName"],
            "Tier": s.get("tierLevel"),
            "Risk Index": s.get("riskIndex"),
            "Facility": f["facilityName"],
            "Country": loc.get("countryName"),
            "TTR (days)": f.get("timeToRecover"),
            "TTS (days)": f.get("timeToSurvive"),
            "Dual Sourced": f.get("isDualSourced"),
        })

if rows:
    df_suppliers = pd.DataFrame(rows)
    display(df_suppliers)
else:
    print("No supplier data extracted.")

# ── Products table ────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("📦 Products")
print("=" * 60)
if products:
    df_products = pd.DataFrame([{
        "Product": p["productName"],
        "Lead Time (days)": p.get("leadTime"),
        "OTD Rate (%)": p.get("onTimeDeliveryRate"),
        "Dual Sourced": p.get("isDualSourced"),
    } for p in products])
    display(df_products)
else:
    print("No product data extracted.")

# ── Risk events table ─────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("⚠️  Risk Events")
print("=" * 60)
if risk_events:
    df_risk = pd.DataFrame([{
        "Event": r["eventName"],
        "Country": r.get("affectsCountry"),
        "Revenue Impact ($)": r.get("revenueImpact"),
        "Time to Awareness (days)": r.get("timeToAwareness"),
        "Time to Action (days)": r.get("timeToAction"),
    } for r in risk_events])
    display(df_risk)
else:
    print("No risk event data extracted.")

# ── NetworkX graph stats ──────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("🕸️  Graph Statistics")
print("=" * 60)

G = nx.DiGraph()
for s in suppliers:
    G.add_node(s["supplierName"], type="Supplier")
    for f in s.get("owns_facilities", []):
        G.add_node(f["facilityName"], type="Facility")
        G.add_edge(s["supplierName"], f["facilityName"], relation="OWNS")
        loc = f.get("locatedIn") or {}
        if loc.get("countryName"):
            G.add_node(loc["countryName"], type="Location")
            G.add_edge(f["facilityName"], loc["countryName"], relation="LOCATED_IN")

for r in risk_events:
    G.add_node(r["eventName"], type="RiskEvent")

print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.4f}")

if G.number_of_nodes() > 0:
    degree_sorted = sorted(G.degree(), key=lambda x: x[1], reverse=True)
    print(f"\n  Top 10 nodes by degree:")
    for name, deg in degree_sorted[:10]:
        node_type = G.nodes[name].get("type", "")
        print(f"    {name[:40]:40s} [{node_type}] degree={deg}")

## 12. Download Results

In [ ]:
import shutil
from pathlib import Path

sh = shutil
sh.copy("ontology/resilience_ontology.ttl", "skgb_output/")
sh.copy("ontology/resilience_shapes.ttl", "skgb_output/")

archive_path = shutil.make_archive("skgb_results", "zip", ".", "skgb_output")
print(f"✅ Archive created: {archive_path}")
print(f"📁 Find it at: {Path(archive_path).absolute()}")

## 13. Stop Ollama

In [ ]:
try:
    if ollama_proc is not None:
        ollama_proc.terminate()
        ollama_proc.wait()
        print("✅ Ollama stopped.")
    else:
        print("✅ Ollama running via app — stop it manually if needed.")
except NameError:
    print("Ollama proc not found — already stopped or never started.")

---
## Evaluation Section
The cells below are for gold set annotation and ablation study evaluation only. They are not part of the main pipeline run.

---
### 14. Gold Set — Chunk Selection

In [ ]:
import json
from pathlib import Path

all_chunks = json.loads(Path("./forked_kg_construction/skgb_output/chunks_output/all_chunks.json").read_text())

# Find content-rich chunks from Dixon and Samsung
candidates = []
for i, chunk in enumerate(all_chunks):
    doc = chunk.get("metadata", {}).get("doc_name", "")
    content = chunk.get("content", "")
    section = chunk.get("section_title", "N/A")
    word_count = len(content.split())
    
    if ("Dixon" in doc or "Samsung" in doc or "Apple" in doc) and word_count >= 100:
        candidates.append({
            "index": i,
            "doc": doc,
            "section": section,
            "words": word_count,
            "preview": content[:200]
        })

# Sort by word count descending and show top 20
candidates.sort(key=lambda x: x["words"], reverse=True)
for c in candidates[:20]:
    print(f"[{c['index']}] {c['doc'][:40]} | {c['section'][:40]} | {c['words']} words")
    print(f"  {c['preview'][:150]}...")
    print()

### 15. Gold Set — Print First Candidate Chunks

In [ ]:
target_indices = [1656, 1665, 693, 731, 1670]

for idx in target_indices:
    chunk = all_chunks[idx]
    print(f"{'='*60}")
    print(f"INDEX: {idx}")
    print(f"DOC: {chunk.get('metadata', {}).get('doc_name', 'N/A')}")
    print(f"SECTION: {chunk.get('section_title', 'N/A')}")
    print(f"CONTENT:")
    print(chunk.get('content', ''))
    print()

### 16. Gold Set — Find Best Chunks by Keyword

In [ ]:
target_keywords = ['India', 'Vietnam', 'supplier', 'facility', 'TTR', 
                   'lead time', 'dual sourc', 'manufacturing', 'sourcing']

good_chunks = []
for i, chunk in enumerate(all_chunks):
    content = chunk.get("content", "").lower()
    doc = chunk.get("metadata", {}).get("doc_name", "")
    word_count = len(content.split())
    
    keyword_hits = sum(1 for k in target_keywords if k.lower() in content)
    
    if keyword_hits >= 3 and word_count >= 100:
        good_chunks.append({
            "index": i,
            "doc": doc[:40],
            "section": chunk.get("section_title", "N/A")[:40],
            "words": word_count,
            "hits": keyword_hits,
            "preview": chunk.get("content", "")[:300]
        })

good_chunks.sort(key=lambda x: x["hits"], reverse=True)
for c in good_chunks[:15]:
    print(f"[{c['index']}] {c['doc']} | {c['section']} | {c['words']}w | {c['hits']} keywords")
    print(f"  {c['preview'][:200]}...")
    print()

### 17. Gold Set — Print Final Gold Chunks

In [ ]:
target_indices = [380, 77, 2466]

for idx in target_indices:
    chunk = all_chunks[idx]
    print(f"{'='*60}")
    print(f"INDEX: {idx}")
    print(f"DOC: {chunk.get('metadata', {}).get('doc_name', 'N/A')}")
    print(f"SECTION: {chunk.get('section_title', 'N/A')}")
    print(f"CONTENT:")
    print(chunk.get('content', ''))
    print()

### 18. Gold Set — Claude Extraction and P/R/F1

In [ ]:
gold_indices = [380, 77, 2466]
claude_results = {}

for idx in gold_indices:
    chunk = all_chunks[idx]
    text = chunk.get("content", "")
    result = extraction_chain.invoke({"text": text})
    claude_results[idx] = result
    print(f"\n{'='*60}")
    print(f"CHUNK {idx}")
    print(result.model_dump_json(indent=2))

### 19. Ablation — Token-aware Chunking Generation

In [ ]:
import json
from pathlib import Path

# ── Token-aware chunking (Ablation 1) ────────────────────────────────────────
# Splits markdown files into fixed-size windows of ~600 tokens
# ignoring document structure — baseline comparison vs semantic chunking

def token_aware_chunk(text, doc_name, tokens_per_chunk=600, overlap_tokens=50):
    """Split text into fixed-size token windows."""
    words = text.split()
    chunks = []
    step = tokens_per_chunk - overlap_tokens
    
    for start in range(0, len(words), step):
        end = min(start + tokens_per_chunk, len(words))
        chunk_text = " ".join(words[start:end])
        chunks.append({
            "content": chunk_text,
            "section_title": f"Token chunk {start//step + 1}",
            "metadata": {"doc_name": doc_name}
        })
        if end == len(words):
            break
    
    return chunks

# Load all markdown files from Docling output
md_dir = Path("skgb_output/build_docling")
all_token_chunks = []

for md_file in sorted(md_dir.glob("*.md")):
    text = md_file.read_text(encoding="utf-8")
    doc_name = md_file.stem
    chunks = token_aware_chunk(text, doc_name)
    all_token_chunks.extend(chunks)
    print(f"  {md_file.name}: {len(chunks)} token-aware chunks")

print(f"\n📦 Total token-aware chunks: {len(all_token_chunks)}")
print(f"📦 Total semantic chunks:    {len(all_chunks)}")
print(f"\nDifference: {len(all_token_chunks) - len(all_chunks):+d} chunks")

# Save for reference
token_chunks_path = Path("skgb_output/chunks_output/all_chunks_token_aware.json")
token_chunks_path.write_text(
    json.dumps(all_token_chunks, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
print(f"\n✅ Token-aware chunks saved to {token_chunks_path}")

### 20. Ablation — Token-aware Matched Chunk Extraction

In [ ]:
import time

# ── Find the token-aware equivalents of gold chunks 380, 77, 2466 ─────────────
# Since token chunking splits differently, find chunks covering the same content
# by matching the gold chunk text against token chunks

gold_texts = {
    380: all_chunks[380]["content"][:200],   # first 200 chars as fingerprint
    77: all_chunks[77]["content"][:200],
    2466: all_chunks[2466]["content"][:200],
}

token_gold_chunks = {}
for gold_idx, fingerprint in gold_texts.items():
    # Find the token chunk whose content overlaps most with the gold chunk
    best_match = None
    best_overlap = 0
    fingerprint_words = set(fingerprint.lower().split())
    
    for i, tc in enumerate(all_token_chunks):
        tc_words = set(tc["content"].lower().split())
        overlap = len(fingerprint_words & tc_words)
        if overlap > best_overlap:
            best_overlap = overlap
            best_match = (i, tc)
    
    if best_match:
        token_gold_chunks[gold_idx] = best_match
        print(f"Gold chunk {gold_idx} → Token chunk {best_match[0]} "
              f"(overlap: {best_overlap} words)")
        print(f"  Preview: {best_match[1]['content'][:150]}...")
        print()

# ── Run Claude on matched token chunks ───────────────────────────────────────
print("\nRunning Claude on token-aware gold chunks...\n")
token_results = {}

for gold_idx, (token_idx, token_chunk) in token_gold_chunks.items():
    text = token_chunk["content"]
    result = extraction_chain.invoke({"text": text})
    token_results[gold_idx] = result
    print(f"Chunk {gold_idx} → {len(result.suppliers)} suppliers, "
          f"{len(result.products)} products, "
          f"{len(result.risk_events)} risk events")
    time.sleep(1.0)

print("\n✅ Token-aware extraction complete")

### 21. Ablation — Semantic vs Token-aware Comparison Table

In [ ]:
# ── Ablation comparison: Semantic vs Token-aware ──────────────────────────────

# Gold annotations (from manual review)
gold = {
    380: {"suppliers": 4, "facilities": 1, "products": 7, "risk_events": 2},
    77:  {"suppliers": 3, "facilities": 4, "products": 8, "risk_events": 0},
    2466:{"suppliers": 1, "facilities": 0, "products": 12, "risk_events": 2},
}

# Improved semantic results (from previous evaluation)
semantic = {
    380: {"suppliers": 3, "facilities": 3, "products": 7, "risk_events": 1},
    77:  {"suppliers": 0, "facilities": 0, "products": 7, "risk_events": 0},
    2466:{"suppliers": 1, "facilities": 0, "products": 12, "risk_events": 2},
}

def compute_f1(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    return round(p, 2), round(r, 2), round(f1, 2)

print("=" * 70)
print("ABLATION STUDY — Semantic vs Token-aware Chunking")
print("=" * 70)

for entity_type in ["suppliers", "products", "risk_events"]:
    print(f"\n── {entity_type.upper()} ──")
    print(f"{'Chunk':<8} {'Gold':<6} {'Semantic':<10} {'Token':<10} {'Sem F1':<10} {'Tok F1':<10}")
    print("-" * 60)
    
    sem_total_tp = sem_total_fp = sem_total_fn = 0
    tok_total_tp = tok_total_fp = tok_total_fn = 0
    
    for idx in [380, 77, 2466]:
        g = gold[idx][entity_type]
        s = semantic[idx][entity_type]
        
        # Token-aware results
        if entity_type == "suppliers":
            t = len(token_results[idx].suppliers)
        elif entity_type == "products":
            t = len(token_results[idx].products)
        else:
            t = len(token_results[idx].risk_events)
        
        # Compute TP/FP/FN (simplified — count-based approximation)
        sem_tp = min(s, g)
        sem_fp = max(0, s - g)
        sem_fn = max(0, g - s)
        
        tok_tp = min(t, g)
        tok_fp = max(0, t - g)
        tok_fn = max(0, g - t)
        
        sem_total_tp += sem_tp; sem_total_fp += sem_fp; sem_total_fn += sem_fn
        tok_total_tp += tok_tp; tok_total_fp += tok_fp; tok_total_fn += tok_fn
        
        _, _, sem_f1 = compute_f1(sem_tp, sem_fp, sem_fn)
        _, _, tok_f1 = compute_f1(tok_tp, tok_fp, tok_fn)
        
        print(f"{idx:<8} {g:<6} {s:<10} {t:<10} {sem_f1:<10} {tok_f1:<10}")
    
    _, _, sem_overall = compute_f1(sem_total_tp, sem_total_fp, sem_total_fn)
    _, _, tok_overall = compute_f1(tok_total_tp, tok_total_fp, tok_total_fn)
    print(f"{'TOTAL':<8} {'':6} {'':10} {'':10} {sem_overall:<10} {tok_overall:<10}")

print("\n" + "=" * 70)
print(f"Chunking strategy comparison:")
print(f"  Semantic chunking total F1:    see above")
print(f"  Token-aware chunking total F1: see above")
print(f"\nChunk count comparison:")
print(f"  Semantic:    {len(all_chunks)} chunks")
print(f"  Token-aware: {len(all_token_chunks)} chunks")